In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler

# 1. โหลดข้อมูลและระบุ Features (25 ตัวแปร)
df = pd.read_csv('Landslide_Final_Cleaned_V2.csv')
TARGET_COL = 'Geohaz_E'

features_to_use = [
    'CHIRPS_Day_1', 'CHIRPS_Day_2', 'CHIRPS_Day_3', 'CHIRPS_Day_4', 'CHIRPS_Day_5',
    'CHIRPS_Day_6', 'CHIRPS_Day_7', 'CHIRPS_Day_8', 'CHIRPS_Day_9', 'CHIRPS_Day_10',
    'Elevation_Extracted', 'Slope_Extracted', 'Aspect_Extracted',
    'MODIS_LC', 'NDVI', 'NDWI', 'TWI', 'Soil_Type',
    'Road_Zone',
    'Rain_3D_Prior', 'Rain_5D_Prior', 'Rain_7D_Prior',
    'Rain3D_x_Slope', 'Rain5D_x_Slope', 'Rain7D_x_Slope'
]

X = df[features_to_use].fillna(df[features_to_use].median(numeric_only=True))
y = df[TARGET_COL]

# 2. แก้ปัญหา Data Imbalance (Oversampling)
df_combined = pd.concat([X, y], axis=1)
majority = df_combined[df_combined[TARGET_COL] == 0.0]
minority = df_combined[df_combined[TARGET_COL] == 1.0]
minority_upsampled = minority.sample(n=len(majority), replace=True, random_state=42)
df_balanced = pd.concat([majority, minority_upsampled])

X_bal = df_balanced[features_to_use]
y_bal = df_balanced[TARGET_COL]

# 3. แบ่งข้อมูลและทำ Normalization
X_train, X_test, y_train, y_test = train_test_split(X_bal, y_bal, test_size=0.3, random_state=42, stratify=y_bal)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# เซฟ Scaler ไว้ใช้งานจริง
joblib.dump(scaler, 'landslide_scaler.pkl')

# 4. เทรนโมเดลเปรียบเทียบ
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

best_recall = 0
best_model = None
winner_name = ""

# เตรียมกราฟ
fig_fi, axes_fi = plt.subplots(2, 2, figsize=(16, 12))
fig_cm, axes_cm = plt.subplots(2, 2, figsize=(12, 10))
axes_fi = axes_fi.flatten()
axes_cm = axes_cm.flatten()

for i, (name, model) in enumerate(models.items()):
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    # คำนวณ Recall เพื่อหาผู้ชนะ
    curr_recall = recall_score(y_test, y_pred)
    if curr_recall > best_recall:
        best_recall = curr_recall
        best_model = model
        winner_name = name

    # กราฟที่ 2: Feature Importance
    if hasattr(model, 'feature_importances_'):
        imp = model.feature_importances_
    else:
        imp = np.abs(model.coef_[0])

    imp_series = pd.Series(imp, index=features_to_use).sort_values(ascending=False).head(10)
    sns.barplot(x=imp_series.values, y=imp_series.index, ax=axes_fi[i], palette='mako')
    axes_fi[i].set_title(f'Feature Importance: {name}')

    # กราฟที่ 3: Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No', 'Yes'])
    disp.plot(ax=axes_cm[i], cmap='Blues', values_format='d')
    axes_cm[i].set_title(f'CM: {name} (Recall: {curr_recall:.2f})')

# 5. เซฟโมเดลที่ดีที่สุด (Highest Recall)
joblib.dump(best_model, f'best_model_{winner_name.replace(" ", "_")}_Recall.pkl')

plt.tight_layout()
plt.show()

print(f"🏆 ผู้ชนะคือ {winner_name} ด้วยค่า Recall {best_recall*100:.2f}%")